# Второй этап экспиремнто

На данный момент есть оптимальный вариант: blend_oof = alpha * oof_cascade + (1 - alpha) * oof_global

BEST alpha_E2: 0.9800000000000002

BEST alpha_global: 0.019999999999999796

BEST blend CV WMAE: 66925.48224836333

CatBoost CV WMAE: 66962.77961396411

oof_cascade, scores_cascade, diag_cascade = cv_segmented_catboost(
    segment_col='segment_cascade',
    min_segment_size=1000,
    experiment_name='E2_salary_hdb_dp_cascade'
)

display(diag_cascade)


oof_global, scores_global = cv_global_catboost()

In [ ]:
import os
import re
import json
import zipfile
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 100)
pd.options.display.float_format = '{:,.2f}'.format

RANDOM_STATE = 42
MIN_SEGMENT_SIZE = 1000

DATA_PATH = Path('.')
SUBMIT_PATH = Path('submissions')
MODELS_PATH = Path('models')

SUBMIT_PATH.mkdir(exist_ok=True)
MODELS_PATH.mkdir(exist_ok=True)


import xgboost as xgb
from xgboost import XGBRegressor

print('xgboost version:', xgb.__version__)

xgboost version: 3.3.0


## 1. Загрузка данных

In [62]:
TRAIN_RAW = pd.read_csv(DATA_PATH / 'train.csv', sep=';', decimal=',', index_col=0, low_memory=False)
TEST_RAW = pd.read_csv(DATA_PATH / 'test.csv', sep=';', decimal=',', index_col=0, low_memory=False)

TARGET_COL = 'target'
WEIGHT_COL = 'w'

y_all = TRAIN_RAW[TARGET_COL].astype(float)
w_all = TRAIN_RAW[WEIGHT_COL].astype(float)

print('TRAIN_RAW:', TRAIN_RAW.shape)
print('TEST_RAW:', TEST_RAW.shape)

target_stats = TRAIN_RAW[TARGET_COL].describe().to_frame(name='target')
target_stats.index = [
    'Количество',
    'Среднее',
    'Стандартное отклонение',
    'Минимум',
    '25% квартиль',
    'Медиана',
    '75% квартиль',
    'Максимум'
]
target_stats['target'] = target_stats['target'].apply(lambda x: f'{x:,.0f}'.replace(',', ' '))
display(target_stats)

TRAIN_RAW: (76786, 223)
TEST_RAW: (73214, 221)


,target
Количество,76 786
Среднее,92 648
Стандартное отклонение,112 409
Минимум,20 000
25% квартиль,39 710
Медиана,62 754
75% квартиль,100 202
Максимум,1 500 000


In [63]:
def WMAE(y_true, y_pred, weights):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    weights = np.asarray(weights, dtype=float)
    return np.sum(weights * np.abs(y_true - y_pred)) / (np.sum(weights) + 1e-12)


def weighted_median(values, weights):
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    order = np.argsort(values)
    values_sorted = values[order]
    weights_sorted = weights[order]
    cumsum = np.cumsum(weights_sorted)
    cutoff = weights_sorted.sum() / 2
    return values_sorted[np.searchsorted(cumsum, cutoff)]


wm = weighted_median(y_all.values, w_all.values)
baseline_pred = np.full(len(y_all), wm)

print('weighted median:', f'{wm:,.0f}'.replace(',', ' '))
print('baseline train WMAE:', f'{WMAE(y_all.values, baseline_pred, w_all.values):,.2f}'.replace(',', ' '))
print('NaN target:', y_all.isna().sum())
print('NaN weight:', w_all.isna().sum())
print('weight <= 0:', int((w_all <= 0).sum()))

weighted median: 84 024
baseline train WMAE: 131 764.18
NaN target: 0
NaN weight: 0
weight <= 0: 0


## 2. Предобработка

In [64]:
def add_date_features(TRAIN, TEST):
    TRAIN = TRAIN.copy()
    TEST = TEST.copy()

    common_cols = [c for c in TRAIN.columns if c in TEST.columns]
    date_cols = [
        c for c in common_cols
        if c == 'dt' or 'date' in c.lower() or c.lower().endswith('_dt')
    ]

    for col in date_cols:
        tr_dt = pd.to_datetime(TRAIN[col], errors='coerce')
        te_dt = pd.to_datetime(TEST[col], errors='coerce')

        if tr_dt.notna().mean() < 0.05 and te_dt.notna().mean() < 0.05:
            continue

        for df, dt in [(TRAIN, tr_dt), (TEST, te_dt)]:
            df[f'{col}_month'] = dt.dt.month
            df[f'{col}_quarter'] = dt.dt.quarter
            df[f'{col}_year'] = dt.dt.year
            df[f'{col}_dayofweek'] = dt.dt.dayofweek

        TRAIN = TRAIN.drop(columns=[col])
        TEST = TEST.drop(columns=[col])

        print('date features added:', col)

    return TRAIN, TEST


In [65]:
def try_convert_object_numeric(TRAIN, TEST, min_success_rate=0.98):
    TRAIN = TRAIN.copy()
    TEST = TEST.copy()

    common_cols = [c for c in TRAIN.columns if c in TEST.columns]
    obj_cols = TRAIN[common_cols].select_dtypes(include=['object']).columns.tolist()
    converted = []

    for col in obj_cols:
        parsed_train = pd.to_numeric(TRAIN[col].astype(str).str.replace(',', '.', regex=False), errors='coerce')
        mask = TRAIN[col].notna()

        if mask.sum() > 0 and parsed_train[mask].notna().mean() >= min_success_rate:
            TRAIN[col] = parsed_train
            TEST[col] = pd.to_numeric(TEST[col].astype(str).str.replace(',', '.', regex=False), errors='coerce')
            converted.append(col)

    print('object -> numeric converted:', len(converted))
    return TRAIN, TEST

In [66]:
def add_group_notna_counts(TRAIN, TEST):
    TRAIN = TRAIN.copy()
    TEST = TEST.copy()

    common_cols = [c for c in TRAIN.columns if c in TEST.columns]

    groups = {
        'dp': [c for c in common_cols if c.startswith('dp_')],
        'hdb': [c for c in common_cols if c.lower().startswith('hdb') or 'bki' in c.lower() or 'bureau' in c.lower()],
        'salary': [c for c in common_cols if 'salary' in c.lower()],
        'income': [c for c in common_cols if 'income' in c.lower()],
        'loan': [c for c in common_cols if 'loan' in c.lower() or 'credit' in c.lower()],
        'card': [c for c in common_cols if 'card' in c.lower()],
        'cashflow': [c for c in common_cols if 'cashflow' in c.lower() or 'turn' in c.lower()],
        'balance': [c for c in common_cols if 'balance' in c.lower() or 'bal' in c.lower()],
        'operation': [c for c in common_cols if 'oper' in c.lower() or 'purch' in c.lower() or 'payment' in c.lower() or 'transaction' in c.lower()],
    }

    for group_name, cols in groups.items():
      if cols:
          TRAIN[f'{group_name}_notna_cnt'] = TRAIN[cols].notna().sum(axis=1).astype(int)
          TEST[f'{group_name}_notna_cnt'] = TEST[cols].notna().sum(axis=1).astype(int)
      else:
          TRAIN[count_col] = int(0)
          TEST[count_col] = int(0)
      print(f'{group_name} cols:', len(cols))

    return TRAIN, TEST


In [67]:
def add_income_salary_interactions(TRAIN, TEST):
    TRAIN = TRAIN.copy()
    TEST = TEST.copy()

    salary_col = 'salary_6to12m_avg'
    income_col = 'incomeValue'

    salary_exists = (
        salary_col in TRAIN.columns
        and salary_col in TEST.columns
    )

    income_exists = (
        income_col in TRAIN.columns
        and income_col in TEST.columns
    )

    if income_exists:
        TRAIN['has_telecom_income'] = (
            TRAIN[income_col]
            .notna()
            .astype('int8')
        )

        TEST['has_telecom_income'] = (
            TEST[income_col]
            .notna()
            .astype('int8')
        )

    if salary_exists and income_exists:
        train_denominator = (
            TRAIN[income_col]
            .abs()
            .clip(lower=1_000)
        )

        test_denominator = (
            TEST[income_col]
            .abs()
            .clip(lower=1_000)
        )

        TRAIN['salary_to_incomeValue_ratio'] = (
            TRAIN[salary_col] / train_denominator
        ).clip(-20, 20)

        TEST['salary_to_incomeValue_ratio'] = (
            TEST[salary_col] / test_denominator
        ).clip(-20, 20)

        TRAIN['salary_minus_incomeValue'] = (
            TRAIN[salary_col] - TRAIN[income_col]
        )

        TEST['salary_minus_incomeValue'] = (
            TEST[salary_col] - TEST[income_col]
        )

        TRAIN['salary_income_abs_diff'] = (
            TRAIN['salary_minus_incomeValue']
            .abs()
        )

        TEST['salary_income_abs_diff'] = (
            TEST['salary_minus_incomeValue']
            .abs()
        )

        train_max_income = (
            pd.concat(
                [
                    TRAIN[salary_col].abs(),
                    TRAIN[income_col].abs(),
                ],
                axis=1
            )
            .max(axis=1)
            .clip(lower=1_000)
        )

        test_max_income = (
            pd.concat(
                [
                    TEST[salary_col].abs(),
                    TEST[income_col].abs(),
                ],
                axis=1
            )
            .max(axis=1)
            .clip(lower=1_000)
        )

        TRAIN['salary_income_relative_gap'] = (
            TRAIN['salary_income_abs_diff']
            / train_max_income
        ).clip(0, 20)

        TEST['salary_income_relative_gap'] = (
            TEST['salary_income_abs_diff']
            / test_max_income
        ).clip(0, 20)

        TRAIN['salary_income_conflict'] = (
            TRAIN['salary_income_relative_gap'] > 0.5
        ).astype(int)

        TEST['salary_income_conflict'] = (
            TEST['salary_income_relative_gap'] > 0.5
        ).astype(int)

        TRAIN['income_hybrid'] = (
            TRAIN[salary_col]
            .fillna(TRAIN[income_col])
        )

        TEST['income_hybrid'] = (
            TEST[salary_col]
            .fillna(TEST[income_col])
        )

    return TRAIN, TEST

In [68]:
TRAIN, TEST = try_convert_object_numeric(TRAIN_RAW, TEST_RAW)
TRAIN, TEST = add_date_features(TRAIN, TEST)
TRAIN, TEST = add_group_notna_counts(TRAIN, TEST)
TRAIN, TEST = add_income_salary_interactions(TRAIN, TEST)

print('TRAIN:', TRAIN.shape)
print('TEST:', TEST.shape)

object -> numeric converted: 34
date features added: dt
dp cols: 34
hdb cols: 39
salary cols: 8
income cols: 13
loan cols: 10
card cols: 1
cashflow cols: 43
balance cols: 5
operation cols: 32
TRAIN: (76786, 242)
TEST: (73214, 240)


## 3. Каскадная сегментация

In [71]:
def add_base_segments(TRAIN, TEST):
    TRAIN = TRAIN.copy()
    TEST = TEST.copy()

    salary_col = 'salary_6to12m_avg'

    if salary_col in TRAIN.columns and salary_col in TEST.columns:
        TRAIN['has_salary_data'] = TRAIN[salary_col].notna().astype(int)
        TEST['has_salary_data'] = TEST[salary_col].notna().astype(int)
    else:
        TRAIN['has_salary_data'] = (TRAIN['salary_notna_cnt'] > 0).astype(int)
        TEST['has_salary_data'] = (TEST['salary_notna_cnt'] > 0).astype(int)

    TRAIN['has_digital_profile'] = (TRAIN['dp_notna_cnt'] > 0).astype(int)
    TEST['has_digital_profile'] = (TEST['dp_notna_cnt'] > 0).astype(int)

    TRAIN['has_hdb'] = (TRAIN['hdb_notna_cnt'] > 0).astype(int)
    TEST['has_hdb'] = (TEST['hdb_notna_cnt'] > 0).astype(int)

    TRAIN['segment_salary'] = np.where(TRAIN['has_salary_data'] == 1, 'salary', 'no_salary')
    TEST['segment_salary'] = np.where(TEST['has_salary_data'] == 1, 'salary', 'no_salary')

    def cascade(df):
        return np.select(
            [
                df['has_salary_data'] == 1,
                (df['has_salary_data'] == 0) & (df['has_digital_profile'] == 1) & (df['has_hdb'] == 1),
                (df['has_salary_data'] == 0) & (df['has_digital_profile'] == 0) & (df['has_hdb'] == 1),
            ],
            [
                'salary',
                'no_salary_dp_hdb',
                'no_salary_no_dp_hdb',
            ],
            default='no_salary_no_hdb'
        )

    TRAIN['segment_cascade'] = cascade(TRAIN)
    TEST['segment_cascade'] = cascade(TEST)

    return TRAIN, TEST


TRAIN, TEST = add_base_segments(TRAIN, TEST)

display(TRAIN['segment_cascade'].value_counts().to_frame('train_count'))
display(TEST['segment_cascade'].value_counts().to_frame('test_count'))

,train_count
segment_cascade,
no_salary_no_dp_hdb,43322
salary,14875
no_salary_dp_hdb,12378
no_salary_no_hdb,6211


,test_count
segment_cascade,
no_salary_no_dp_hdb,40642
no_salary_dp_hdb,18462
salary,7634
no_salary_no_hdb,6476


### Дополнительное разделение большого сегмента

Большой сегмент `no_salary_no_dp_hdb` делим на две части по заполненности cashflow-признаков:

- `cashflow_high`
- `cashflow_low`

Это финальная сегментация для обучения: `segment_cascade_v2_cashflow_binary`.

In [ ]:
# def split_big_segment_by_cashflow(
#     TRAIN,
#     TEST,
#     base_col='segment_cascade',
#     new_col='segment_cascade_2',
#     big_segment='no_salary_no_dp_hdb',
#     split_col='cashflow_notna_cnt',
#     q_high=0.66
# ):
#     TRAIN = TRAIN.copy()
#     TEST = TEST.copy()

#     TRAIN[new_col] = TRAIN[base_col].astype(str)
#     TEST[new_col] = TEST[base_col].astype(str)

#     train_mask = TRAIN[base_col] == big_segment
#     test_mask = TEST[base_col] == big_segment

#     threshold = TRAIN.loc[train_mask, split_col].fillna(-1).quantile(q_high)

#     TRAIN.loc[train_mask, new_col] = np.where(
#         TRAIN.loc[train_mask, split_col].fillna(-1) > threshold,
#         big_segment + '_cashflow_high',
#         big_segment + '_cashflow_low'
#     )

#     TEST.loc[test_mask, new_col] = np.where(
#         TEST.loc[test_mask, split_col].fillna(-1) > threshold,
#         big_segment + '_cashflow_high',
#         big_segment + '_cashflow_low'
#     )

#     print('cashflow threshold:', threshold)

#     return TRAIN, TEST


# TRAIN, TEST = split_big_segment_by_cashflow(TRAIN, TEST)

# display(TRAIN['segment_cascade_2'].value_counts().to_frame('train_count'))
# display(TEST['segment_cascade_2'].value_counts().to_frame('test_count'))

cashflow threshold: 28.0


,train_count
segment_cascade_2,
no_salary_no_dp_hdb_cashflow_low,29843
salary,14875
no_salary_no_dp_hdb_cashflow_high,13479
no_salary_dp_hdb,12378
no_salary_no_hdb,6211


,test_count
segment_cascade_2,
no_salary_no_dp_hdb_cashflow_low,27326
no_salary_dp_hdb,18462
no_salary_no_dp_hdb_cashflow_high,13316
salary,7634
no_salary_no_hdb,6476


## 4. Подготовка признаков


In [73]:
print('Полных дублей:', TRAIN.duplicated().sum())

TRAIN = TRAIN.drop_duplicates().copy()

Полных дублей: 0


In [74]:
TRAIN['target'].describe(
    percentiles=[0.9, 0.95, 0.99, 0.995, 0.999]
)

,target
count,"76,786.00"
mean,"92,648.24"
std,"112,408.98"
min,"20,000.00"
50%,"62,754.13"
90%,"171,206.68"
95%,"256,074.61"
99%,"596,585.06"
99.5%,"819,156.91"
99.9%,"1,291,317.06"


In [75]:
invalid_mask = (
    (TRAIN['age'].notna() & ~TRAIN['age'].between(14, 100))
    | (TRAIN['incomeValue'].notna() & ~np.isfinite(TRAIN['incomeValue']))
)

print('Подозрительных строк:', invalid_mask.sum())

Подозрительных строк: 0


In [76]:
TRAIN = TRAIN.reset_index(drop=True)
TEST = TEST.reset_index(drop=True)

y = TRAIN['target'].astype(float).copy()
w = TRAIN['w'].astype(float).copy()

print(TRAIN.shape, TEST.shape)
print(y.shape, w.shape)

assert len(TRAIN) == len(y) == len(w)
assert 'target' in TRAIN.columns
assert 'w' in TRAIN.columns

(76786, 248) (73214, 246)
(76786,) (76786,)


In [77]:
exclude_cols = {
    'target',
    'w',
    'segment_cascade',
    'segment_cascade_2',
    'segment_salary',
    'has_dp_data',
    'has_hdb_data',
}

feature_cols = [
    col
    for col in TRAIN.columns
    if col in TEST.columns
    and col not in exclude_cols
]

print('Количество признаков:', len(feature_cols))

Количество признаков: 243


In [42]:
def prepare_X_for_xgb(TRAIN, TEST, feature_cols):
    X_train = TRAIN[feature_cols].copy()
    X_test = TEST[feature_cols].copy()

    X_train = X_train.replace([np.inf, -np.inf], np.nan)
    X_test = X_test.replace([np.inf, -np.inf], np.nan)

    categorical_cols = X_train.select_dtypes(
        include=['object', 'category', 'bool', 'string']
    ).columns.tolist()

    category_mappings = {}

    for col in categorical_cols:
        train_values = (
            X_train[col]
            .astype('string')
            .fillna('__MISSING__')
        )

        test_values = (
            X_test[col]
            .astype('string')
            .fillna('__MISSING__')
        )

        # Создаём единое пространство категорий для TRAIN и TEST,
        # чтобы одинаковые значения получили одинаковые коды.
        all_values = pd.concat(
            [train_values, test_values],
            axis=0
        )

        unique_values = pd.Index(all_values.unique())

        mapping = {
            value: code
            for code, value in enumerate(unique_values)
        }

        X_train[col] = (
            train_values
            .map(mapping)
            .fillna(-1)
            .astype('int32')
        )

        X_test[col] = (
            test_values
            .map(mapping)
            .fillna(-1)
            .astype('int32')
        )

        category_mappings[col] = mapping

    bool_cols = X_train.select_dtypes(include=['bool']).columns

    for col in bool_cols:
        X_train[col] = X_train[col].astype('int8')
        X_test[col] = X_test[col].astype('int8')

    print('Категориальных колонок закодировано:', len(categorical_cols))

    return (
        X_train,
        X_test,
        categorical_cols,
        category_mappings,
    )

In [78]:
X_xgb, X_test_xgb, categorical_cols, category_mappings = (
    prepare_X_for_xgb(
        TRAIN,
        TEST,
        feature_cols
    )
)

assert len(X_xgb) == len(TRAIN)
assert len(X_test_xgb) == len(TEST)
assert list(X_xgb.columns) == list(X_test_xgb.columns)

Категориальных колонок закодировано: 6


In [79]:
y_bins = pd.qcut(
    y.rank(method='first'),
    q=10,
    labels=False
)

stratify_key = (
    TRAIN['segment_cascade'].astype(str)
    + '__'
    + y_bins.astype(str)
)

train_idx, valid_idx = train_test_split(
    np.arange(len(TRAIN)),
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=stratify_key
)

print('train:', len(train_idx))
print('valid:', len(valid_idx))

train: 61428
valid: 15358


## 5. Параметры модели

In [80]:
xgb_params = {
    'n_estimators': 5000,
    'learning_rate': 0.03,
    'max_depth': 7,

    'min_child_weight': 20,
    'subsample': 0.85,
    'colsample_bytree': 0.8,

    'reg_alpha': 0.1,
    'reg_lambda': 5.0,

    'objective': 'reg:absoluteerror',

    'tree_method': 'hist',
    'device': 'cuda',

    'random_state': 42,
}

## 6. Финальное обучение моделей на всём train

In [20]:
def train_valid_segmented_xgb_model(
    X_train_data,
    X_test_data,
    y_train,
    w_train,
    meta_train,
    meta_test,
    params,
    segment_col,
    train_idx,
    valid_idx,
    min_segment_size=1000,
    early_stopping_rounds=100,
    verbose=200,
):

    models = {}

    pred_valid = np.zeros(len(valid_idx), dtype=float)
    pred_test = np.zeros(len(meta_test), dtype=float)
    X_tr_full = (X_train_data.iloc[train_idx].copy())
    X_val_full = (X_train_data.iloc[valid_idx].copy())
    y_tr_full = (y_train.iloc[train_idx])
    y_val_full = (y_train.iloc[valid_idx])
    w_tr_full = (w_train.iloc[train_idx])
    w_val_full = (w_train.iloc[valid_idx])

    meta_tr = (meta_train.iloc[train_idx])
    meta_val = (meta_train.iloc[valid_idx])

    fit_params = params.copy()
    fit_params['eval_metric'] = 'mae'
    fit_params['early_stopping_rounds'] = (early_stopping_rounds)

    print('fit fallback global XGBoost')

    fallback_model = XGBRegressor(**fit_params)

    fallback_model.fit(
        X_tr_full,
        y_tr_full,
        sample_weight=w_tr_full,
        eval_set=[
            (
                X_val_full,
                y_val_full,
            )
        ],
        sample_weight_eval_set=[w_val_full],
        verbose=verbose
    )

    models['fallback_global'] = fallback_model

    all_segments = sorted(
        set(
            meta_train[segment_col]
            .dropna()
            .astype(str)
            .unique()
        )
        |
        set(
            meta_test[segment_col]
            .dropna()
            .astype(str)
            .unique()
        )
    )

# модели по сегментам
    for seg in all_segments:
        tr_mask = (
            meta_tr[segment_col]
            .astype(str)
            .to_numpy()
            == str(seg)
        )

        val_mask = (
            meta_val[segment_col]
            .astype(str)
            .to_numpy()
            == str(seg)
        )

        test_mask = (
            meta_test[segment_col]
            .astype(str)
            .to_numpy()
            == str(seg)
        )

        n_train_seg = int(
            tr_mask.sum()
        )

        n_valid_seg = int(
            val_mask.sum()
        )

        n_test_seg = int(
            test_mask.sum()
        )

        print(
            f'\nsegment: {seg}'
            f' | train: {n_train_seg}'
            f' | valid: {n_valid_seg}'
            f' | test: {n_test_seg}'
        )

        if n_train_seg < min_segment_size:
            print('too small -> fallback')

            if n_valid_seg > 0:
                pred_valid[val_mask] = (
                    fallback_model.predict(
                        X_val_full.iloc[val_mask]
                    )
                )

            if n_test_seg > 0:
                pred_test[test_mask] = (
                    fallback_model.predict(
                        X_test_data.iloc[test_mask]
                    )
                )

            continue

        X_tr_seg = (X_tr_full.iloc[tr_mask].copy())

        X_val_seg = (X_val_full.iloc[val_mask].copy())

        X_test_seg = (X_test_data.iloc[test_mask].copy())

        y_tr_seg = (y_tr_full.iloc[tr_mask])

        y_val_seg = (y_val_full.iloc[val_mask])

        w_tr_seg = (w_tr_full.iloc[tr_mask])

        w_val_seg = (w_val_full.iloc[val_mask])

        constant_cols = [
            col
            for col in X_tr_seg.columns
            if X_tr_seg[col].nunique(
                dropna=False
            ) <= 1
        ]

        if constant_cols:
            X_tr_seg.drop(
                columns=constant_cols,
                inplace=True,
            )

            X_val_seg.drop(
                columns=[
                    col
                    for col in constant_cols
                    if col in X_val_seg.columns
                ],
                inplace=True,
            )

            X_test_seg.drop(
                columns=[
                    col
                    for col in constant_cols
                    if col in X_test_seg.columns
                ],
                inplace=True,
            )

            print('constant features removed:',len(constant_cols))

        models[f'{seg}__features'] = (X_tr_seg.columns.tolist())

        if n_valid_seg > 0:
            segment_params = params.copy()

            segment_params['eval_metric'] = 'mae'
            segment_params[
                'early_stopping_rounds'
            ] = early_stopping_rounds

            model = XGBRegressor(
                **segment_params
            )

            model.fit(
                X_tr_seg,
                y_tr_seg,
                sample_weight=w_tr_seg,

                eval_set=[
                    (
                        X_val_seg,
                        y_val_seg,
                    )
                ],

                sample_weight_eval_set=[
                    w_val_seg
                ],

                verbose=verbose
            )

        else:
            segment_params = params.copy()

            segment_params.pop(
                'early_stopping_rounds',
                None,
            )

            segment_params['eval_metric'] = 'mae'

            model = XGBRegressor(
                **segment_params
            )

            model.fit(
                X_tr_seg,
                y_tr_seg,
                sample_weight=w_tr_seg,
                verbose=False,
            )

        models[seg] = model

        if n_valid_seg > 0:
            pred_valid[val_mask] = (
                model.predict(
                    X_val_seg
                )
            )

        if n_test_seg > 0:
            pred_test[test_mask] = (
                model.predict(
                    X_test_seg
                )
            )

    pred_valid = np.clip(
        pred_valid,
        0,
        None,
    )

    pred_test = np.clip(
        pred_test,
        0,
        None,
    )

    valid_score = WMAE(
        y_val_full.to_numpy(),
        pred_valid,
        w_val_full.to_numpy(),
    )

    print('\nVALID WMAE:',valid_score)

    return (models,pred_valid, pred_test, valid_score)

In [81]:
models_xgb, pred_valid_xgb, pred_test_xgb, score_xgb = (
    train_valid_segmented_xgb_model(
        X_train_data=X_xgb,
        X_test_data=X_test_xgb,
        y_train=y,
        w_train=w,
        meta_train=TRAIN,
        meta_test=TEST,
        params=xgb_params,
        segment_col='segment_cascade',
        train_idx=train_idx,
        valid_idx=valid_idx,
        min_segment_size=1000,
        early_stopping_rounds=100,
        verbose=200,
    )
)

fit fallback global XGBoost
[0]	validation_0-mae:123591.38826
[200]	validation_0-mae:60849.44611
[400]	validation_0-mae:59557.88089
[600]	validation_0-mae:59056.35108
[800]	validation_0-mae:59004.40632
[905]	validation_0-mae:59037.18292

segment: no_salary_dp_hdb | train: 9902 | valid: 2476 | test: 18462
constant features removed: 11
[0]	validation_0-mae:138017.80354
[200]	validation_0-mae:58354.02532
[400]	validation_0-mae:58013.54686
[411]	validation_0-mae:58046.85897


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:553: UserWarning: [12:02:33] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)



segment: no_salary_no_dp_hdb | train: 34658 | valid: 8664 | test: 40642
constant features removed: 46
[0]	validation_0-mae:108682.35373
[200]	validation_0-mae:64296.59915
[400]	validation_0-mae:63766.97066
[457]	validation_0-mae:63827.83612

segment: no_salary_no_hdb | train: 4968 | valid: 1243 | test: 6476
constant features removed: 50
[0]	validation_0-mae:85800.29334
[200]	validation_0-mae:76250.28937
[228]	validation_0-mae:76291.06200

segment: salary | train: 11900 | valid: 2975 | test: 7634
constant features removed: 2
[0]	validation_0-mae:149511.40431
[200]	validation_0-mae:33172.73541
[400]	validation_0-mae:32538.13281
[600]	validation_0-mae:32296.31102
[676]	validation_0-mae:32325.49423

VALID WMAE: 57077.65569719438


In [83]:
def fit_final_xgb_model(X_train, y_train, w_train, params):
    model = XGBRegressor(**params)

    model.fit(
        X_train,
        y_train,
        sample_weight=w_train,
        verbose=False
    )

    return model
def train_final_segmented_xgb_model(
    X_train_data,
    y_train,
    w_train,
    meta_train,
    params,
    segment_col,
    min_segment_size=1000
):
    models = {}

    print('fit fallback global XGBoost model')

    fallback_model = fit_final_xgb_model(
        X_train_data,
        y_train,
        w_train,
        params
    )

    models['fallback_global'] = fallback_model

    for seg in sorted(meta_train[segment_col].unique()):
        train_mask = meta_train[segment_col].values == seg
        n_train_seg = int(train_mask.sum())

        print(f'\nsegment: {seg} | train: {n_train_seg}')

        if n_train_seg < min_segment_size:
            print('too small -> fallback will be used')
            continue

        model_seg = fit_final_xgb_model(
            X_train_data.iloc[train_mask],
            y_train.iloc[train_mask],
            w_train.iloc[train_mask],
            params
        )

        models[seg] = model_seg

    return models

In [87]:
final_models_xgb = train_final_segmented_xgb_model(
    X_train_data=X_xgb,
    y_train=y,
    w_train=w,
    meta_train=TRAIN,
    params=xgb_params,
    segment_col='segment_cascade',
    min_segment_size=MIN_SEGMENT_SIZE
)

fit fallback global XGBoost model

segment: no_salary_dp_hdb | train: 12378

segment: no_salary_no_dp_hdb | train: 43322

segment: no_salary_no_hdb | train: 6211

segment: salary | train: 14875


## 7. Отдельная функция предсказания

In [90]:
def predict_segmented_model(
    models,
    X_data,
    meta_df,
    segment_col,
    clip_zero=True
):
    pred = np.zeros(len(meta_df))

    for seg in sorted(meta_df[segment_col].unique()):
        mask = meta_df[segment_col].values == seg
        model_name = seg if seg in models else 'fallback_global'

        print(f'predict segment: {seg} | objects: {int(mask.sum())} | model: {model_name}')

        pred[mask] = models[model_name].predict(X_data.iloc[mask])

    if clip_zero:
        pred = np.maximum(pred, 0)

    print('\nPrediction stats:')
    display(pd.Series(pred).describe().to_frame('predict'))

    print('NaN:', int(np.isnan(pred).sum()))
    print('Zero:', int((pred == 0).sum()))
    print('Min:', f'{pred.min():,.2f}'.replace(',', ' '))
    print('Max:', f'{pred.max():,.2f}'.replace(',', ' '))

    return pred

In [93]:
pred_test = predict_segmented_model(
    models=final_models_xgb,
    X_data=X_test_xgb,
    meta_df=TEST,
    segment_col='segment_cascade'
)

predict segment: no_salary_dp_hdb | objects: 18462 | model: no_salary_dp_hdb
predict segment: no_salary_no_dp_hdb | objects: 40642 | model: no_salary_no_dp_hdb
predict segment: no_salary_no_hdb | objects: 6476 | model: no_salary_no_hdb
predict segment: salary | objects: 7634 | model: salary

Prediction stats:


,predict
count,"73,214.00"
mean,"96,157.38"
std,"97,351.94"
min,0.00
25%,"42,161.69"
50%,"62,714.69"
75%,"112,617.06"
max,"1,234,658.00"


NaN: 0
Zero: 7
Min: 0.00
Max: 1 234 658.00


## 8. Submit

In [94]:
def make_submit(pred_test, filename='submit_final_cascade.csv'):
    sample_path = DATA_PATH / 'sample_submission.csv'

    if sample_path.exists():
        sub = pd.read_csv(sample_path, sep=';')
        sub = sub[['id']].copy()
    else:
        sub = pd.DataFrame({'id': TEST.index})

    sub['predict'] = pred_test

    assert list(sub.columns) == ['id', 'predict']
    assert len(sub) == len(pred_test)
    assert sub['predict'].isna().sum() == 0

    out_path = SUBMIT_PATH / filename
    sub.to_csv(out_path, sep=';', index=False)

    print('saved:', out_path)
    print('shape:', sub.shape)

    return sub

In [95]:
submit = make_submit(
    pred_test,
    filename='submit_xgb_good.csv'
)

display(submit.head())

saved: submissions/submit_xgb_good.csv
shape: (73214, 2)


,id,predict
0,0,"54,500.01"
1,1,"61,891.05"
2,3,"26,248.01"
3,9,"72,886.80"
4,11,"46,229.91"
